In [ ]:
# --- repo bootstrap ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 04 — Ion Motion Simulation

Simulate ion motion under a moving harmonic well defined by the transport path.

```text
waveform / x_c(t) → ion trajectory x(t) → residual excitation
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig, IonSpecies
from src.ion_transport_waveform.transport_path import minimum_jerk_path
from src.ion_transport_waveform.motion_sim import simulate_ion_motion
from src.ion_transport_waveform.excitation_metrics import residual_amplitude, residual_energy_proxy

cfg = TrapConfig()
ion = IonSpecies()

fig_dir = repo / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)


## 1. Define transport path


In [ ]:
x0 = -160e-6
x1 = 160e-6
duration = 20e-6

t = np.linspace(0, duration, 1200)
path = minimum_jerk_path(t, x0=x0, x1=x1, duration=duration)


## 2. Simulate ion motion


In [ ]:
x_traj, v_traj = simulate_ion_motion(t, path, cfg.omega_rad_s)

print("trajectory length:", len(x_traj))


## 3. Figure: trajectory vs target path


In [ ]:
plt.figure(figsize=(8,4.5))
plt.plot(t*1e6, path*1e6, label="target path")
plt.plot(t*1e6, x_traj*1e6, label="ion trajectory")
plt.xlabel("time (µs)")
plt.ylabel("position (µm)")
plt.title("Ion motion under transport")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "04_motion_trace.png", dpi=180)
plt.show()


## 4. Residual excitation metrics


In [ ]:
amp = residual_amplitude(x_traj, path)
energy = residual_energy_proxy(x_traj, v_traj, path, cfg.omega_rad_s, mass_kg=ion.mass_kg)

print(f"residual amplitude: {amp:.3e} m")
print(f"residual energy proxy: {energy:.3e} J")


## 5. Duration sweep → excitation


In [ ]:
durations = np.array([5,10,20,40,80]) * 1e-6
amps = []

for T in durations:
    tt = np.linspace(0, T, 800)
    path_T = minimum_jerk_path(tt, x0=x0, x1=x1, duration=T)
    xT, vT = simulate_ion_motion(tt, path_T, cfg.omega_rad_s)
    amps.append(residual_amplitude(xT, path_T))

amps = np.array(amps)

for T, a in zip(durations, amps):
    print(f"T={T*1e6:5.1f} µs | residual amp={a:.3e} m")


In [ ]:
plt.figure(figsize=(7.5,4.5))
plt.loglog(durations*1e6, amps, marker="o")
plt.xlabel("transport duration (µs)")
plt.ylabel("residual amplitude (m)")
plt.title("Transport duration vs residual excitation")
plt.tight_layout()
plt.savefig(fig_dir / "04_excitation_vs_duration.png", dpi=180)
plt.show()


## 6. Save results


In [ ]:
np.savez(
    data_dir / "motion_simulation_04.npz",
    t=t,
    path=path,
    x_traj=x_traj,
    v_traj=v_traj,
    residual_amplitude=amp,
    residual_energy=energy
)

print("saved motion simulation data")


## Next

You now have a full pipeline:

```text
basis → well → path → waveform → ion motion → excitation
```
